In [3]:
import pandas as pd
import numpy as np

def compute_metrics(returns, equity, freq="monthly"):
    if freq == "monthly":
        periods = 12
    elif freq == "daily":
        periods = 252

    def rend_cum(returns):
        return np.exp(returns.sum())-1
    
    def rend_annu(returns):
        return (1+rend_cum(returns))**(periods/len(returns))-1
    
    def rend_mean(returns):
        return (1/len(returns))*returns.sum()
        
    def vol_period(returns):
        return returns.std()
    
    def vol_annu(returns):
        return vol_period(returns)*np.sqrt(periods)
    
    def sharpe(returns): 
        return rend_annu(returns)/vol_annu(returns)
    
    def max_drowdown(equity):
        return min((equity-equity.cummax())/equity.cummax())
    
    def sortino(returns):
        return rend_annu(returns)/(vol_annu(returns[returns<0]))
        
    def calmar(returns, equity):
        return rend_annu(returns)/abs(max_drowdown(equity))

    def winrate(returns):
        return (returns>0).sum()/len(returns)

    def var_hist(returns):
        return np.percentile(returns.dropna(), 5)

    def cond_var_hist(returns): 
        return (returns[returns<var_hist(returns)]).mean()
    
    return pd.Series({
        "Cumulative return": f"{rend_cum(returns):.2%}",
        "Annualized return": f"{rend_annu(returns):.2%}",
        "Mean return": f"{rend_mean(returns):.2%}",
        "Period volatility": f"{vol_period(returns):.2%}",
        "Annual volatility": f"{vol_annu(returns):.2%}",
        "Winrate": f"{winrate(returns):.2%}",
        "Sharpe Ratio": sharpe(returns),
        "Max Drawdown": f"{max_drowdown(equity):.2%}",
        "Sortino": sortino(returns),
        "Calmar": calmar(returns, equity),
        "Historic VaR": f"{var_hist(returns):.2%}",
        "Conditional Historic VaR": f"{cond_var_hist(returns):.2%}"
    })

def drawdown(equity):
    return (equity-equity.cummax())/equity.cummax()